In [1]:
import os

import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)

# ---- Paths (strings) ----
REVIEWS_CSV = "../data/raw/reviews.csv"
RECIPES_PKL = "recipes_labeled.pkl"

ALL_PKL = "interactions_all.pkl"
TRAIN_PKL = "interactions_train.pkl"
TEST_PKL = "interactions_test.pkl"
STATS_CSV = "recipe_stats_train.csv"
SUMMARY_CSV = "interactions_summary.csv"

# ---- Decisions ----
LIKE_THRESHOLD = 4        # Liked = 4 stars or more
MIN_EVAL = 5              # Minimum number of interactions for a user to be included in the evaluation
TEST_FRAC = 0.20          # Fraction of liked recipes held out per user for evaluation
SPLIT_MODE = "random"     # "random" (default) or "time" (hold out the most recent liked recipes)
SEED = 42

for p in [REVIEWS_CSV, RECIPES_PKL]:
    assert isinstance(p, str), f"{p!r} must be a string (path string)"
    assert os.path.exists(p), f"Could not find {p}: fix the path"

In [2]:
rev = pd.read_csv(REVIEWS_CSV, low_memory=False)
recipes = pd.read_pickle(RECIPES_PKL)
print(f"reviews: {len(rev):,} | recipes (final): {len(recipes):,}")
print("review columns:", list(rev.columns))

def pick(cands):
    for c in cands:
        if c in rev.columns:
            return c
    return None

COLS = {
    "user": pick(["AuthorId", "user_id", "UserId"]),
    "recipe": pick(["RecipeId", "recipe_id"]),
    "rating": pick(["Rating", "rating"]),
    "date": pick(["DateSubmitted", "date", "DateModified"]),
}
print(COLS)
missing = [k for k in ("user", "recipe", "rating") if COLS[k] is None]
assert not missing, f"Columns not found: {missing}. Update COLS manually"

reviews: 1,401,982 | recipes (final): 42,736
review columns: ['ReviewId', 'RecipeId', 'AuthorId', 'AuthorName', 'Rating', 'Review', 'DateSubmitted', 'DateModified']
{'user': 'AuthorId', 'recipe': 'RecipeId', 'rating': 'Rating', 'date': 'DateSubmitted'}


In [3]:
inter = pd.DataFrame({
    "user_id": pd.to_numeric(rev[COLS["user"]], errors="coerce"),
    "recipe_id": pd.to_numeric(rev[COLS["recipe"]], errors="coerce"),
    "rating_raw": pd.to_numeric(rev[COLS["rating"]], errors="coerce"),
})
inter["date"] = (pd.to_datetime(rev[COLS["date"]], errors="coerce", utc=True)
                 if COLS["date"] else pd.NaT)

funnel = [("raw reviews", len(inter))]
inter = inter.dropna(subset=["user_id", "recipe_id", "rating_raw"])
inter["user_id"] = inter["user_id"].astype("int64")
inter["recipe_id"] = inter["recipe_id"].astype("int64")
funnel.append(("valid user / recipe / rating", len(inter)))

In [4]:
catalog = set(recipes["RecipeId"].astype("int64"))
inter = inter[inter["recipe_id"].isin(catalog)].copy()
funnel.append(("on final recipes", len(inter)))


inter = inter.sort_values(["user_id", "recipe_id", "date"], na_position="first")
inter = inter.drop_duplicates(["user_id", "recipe_id"], keep="last").reset_index(drop=True)
funnel.append(("after removing duplicates", len(inter)))

pd.DataFrame(funnel, columns=["step", "n_interactions"])

,step,n_interactions
0,raw reviews,1401982
1,valid user / recipe / rating,1401982
2,on final recipes,164235
3,after removing duplicates,164235


In [5]:
inter["has_rating"] = inter["rating_raw"].between(1, 5)
inter["rating"] = inter["rating_raw"].where(inter["has_rating"])
inter["liked"] = inter["rating"].ge(LIKE_THRESHOLD).fillna(False).astype(bool)
inter["interaction"] = 1
inter = inter[["user_id", "recipe_id", "rating", "has_rating", "liked", "interaction", "date"]]

print("Raw rating distribution (after merging):")
print(inter["rating"].fillna(0).astype(int).value_counts().sort_index().to_dict(), "(0 = no stars)")
print(f"\nWith stars: {inter['has_rating'].mean()*100:.1f}% | Liked (4+): {inter['liked'].mean()*100:.1f}%")

Raw rating distribution (after merging):
{0: 9383, 1: 2454, 2: 2235, 3: 5745, 4: 25126, 5: 119292} (0 = no stars)

With stars: 94.3% | Liked (4+): 87.9%


In [6]:
n_users = inter["user_id"].nunique()
n_items = inter["recipe_id"].nunique()
per_user = inter.groupby("user_id").size()
per_item = inter.groupby("recipe_id").size()

summary = {
    "interactions": len(inter),
    "users": n_users,
    "recipes_with_interactions": n_items,
    "recipes_in_catalog": len(catalog),
    "catalog_coverage_%": round(n_items / len(catalog) * 100, 1),
    "density_%": round(len(inter) / (n_users * n_items) * 100, 4),
    "median_per_user": float(per_user.median()),
    "median_per_recipe": float(per_item.median()),
    f"users_with_>={MIN_EVAL}": int((per_user >= MIN_EVAL).sum()),
    "five_star_%": round((inter["rating"] == 5).mean() * 100, 1),
    "zero_rating_%": round((~inter["has_rating"]).mean() * 100, 1),
    "liked_%": round(inter["liked"].mean() * 100, 1),
}


label_cols = [c for c in recipes.columns if c.startswith("label_")]
for c in label_cols:
    ids = set(recipes.loc[recipes[c].astype(bool), "RecipeId"].astype("int64"))
    summary[f"{c}_with_interactions_%"] = round(len(ids & set(per_item.index)) / max(len(ids), 1) * 100, 1)

pd.Series(summary, name="value")

interactions                            164235.0000
users                                    60183.0000
recipes_with_interactions                24452.0000
recipes_in_catalog                       42736.0000
catalog_coverage_%                          57.2000
density_%                                    0.0112
median_per_user                              1.0000
median_per_recipe                            2.0000
users_with_>=5                            4854.0000
five_star_%                                 72.6000
zero_rating_%                                5.7000
liked_%                                     87.9000
label_menstrual_with_interactions_%         56.9000
label_follicular_with_interactions_%        56.7000
label_ovulatory_with_interactions_%         56.7000
label_luteal_with_interactions_%            57.7000
Name: value, dtype: float64

In [7]:
print("Number of interactions per user:")
bins = [0, 1, 2, 4, 9, 19, 49, np.inf]
labels = ["1", "2", "3-4", "5-9", "10-19", "20-49", "50+"]
print(pd.cut(per_user, bins=bins, labels=labels).value_counts().sort_index().to_dict())

print("\nNumber of interactions per recipe:")
print(pd.cut(per_item, bins=bins, labels=labels).value_counts().sort_index().to_dict())

top_share = per_item.sort_values(ascending=False)
k = max(1, int(len(top_share) * 0.01))
print(f"\nTop 1% of recipes ({k}) account for {top_share.head(k).sum() / top_share.sum() * 100:.1f}% of interactions (popularity bias)")

Number of interactions per user:
{'1': 45669, '2': 5822, '3-4': 3838, '5-9': 2615, '10-19': 1216, '20-49': 665, '50+': 358}

Number of interactions per recipe:
{'1': 8596, '2': 4353, '3-4': 4354, '5-9': 3909, '10-19': 1846, '20-49': 967, '50+': 427}

Top 1% of recipes (244) account for 26.2% of interactions (popularity bias)


In [8]:
rng = np.random.default_rng(SEED)
n_inter = inter.groupby("user_id")["interaction"].transform("size")
n_liked = inter.groupby("user_id")["liked"].transform("sum")
eval_user = (n_inter >= MIN_EVAL) & (n_liked >= 1)

cand = inter["liked"] & eval_user
if SPLIT_MODE == "time" and inter["date"].notna().any():
    key = -(inter["date"] - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()   # new firstاً
else:
    key = pd.Series(rng.random(len(inter)), index=inter.index)

order = key.where(cand).groupby(inter["user_id"]).rank(method="first")
n_test_user = np.maximum(1, np.round(n_liked * TEST_FRAC)).astype(int)
is_test = cand & (order <= n_test_user)

train = inter[~is_test].reset_index(drop=True)
test = inter[is_test].reset_index(drop=True)

print(f"eval users: {inter.loc[eval_user, 'user_id'].nunique():,}")
print(f"train: {len(train):,} interactions | {train['user_id'].nunique():,} users | {train['recipe_id'].nunique():,} recipes")
print(f"test : {len(test):,} interactions | {test['user_id'].nunique():,} users | {test['recipe_id'].nunique():,} recipes")

eval users: 4,852
train: 146,775 interactions | 60,183 users | 23,328 recipes
test : 17,460 interactions | 4,852 users | 8,337 recipes


In [9]:
key_tr = set(zip(train["user_id"], train["recipe_id"]))
key_te = set(zip(test["user_id"], test["recipe_id"]))
assert not (key_tr & key_te), "Some rows appear in both train and test!"
assert set(test["user_id"]) <= set(train["user_id"]), "A user in the test set has no training data!"
assert test["liked"].all(), "The test set must contain liked recipes only"

cold = ~test["recipe_id"].isin(set(train["recipe_id"]))
print("All checks passed")
print(f"cold test items (no interactions in training): {cold.sum():,} of {len(test):,} ({cold.mean()*100:.1f}%)")
print(f"test interactions per eval user: median {test.groupby('user_id').size().median():.0f}, max {test.groupby('user_id').size().max()}")

All checks passed
cold test items (no interactions in training): 1,209 of 17,460 (6.9%)
test interactions per eval user: median 2, max 174


In [10]:
g = train.groupby("recipe_id")
stats = pd.DataFrame({
    "n_interactions": g.size(),
    "n_ratings": g["has_rating"].sum(),
    "mean_rating": g["rating"].mean(),
    "n_liked": g["liked"].sum(),
})
C_mean = train["rating"].mean()                                    
M_min = float(stats.loc[stats["n_ratings"] > 0, "n_ratings"].quantile(0.75))   
stats["weighted_rating"] = ((stats["n_ratings"] * stats["mean_rating"].fillna(C_mean) + M_min * C_mean)
                            / (stats["n_ratings"] + M_min))
stats = stats.reset_index().sort_values("weighted_rating", ascending=False)
print(f"global mean rating = {C_mean:.3f} | m (75th percentile of n_ratings) = {M_min:.0f}")

names = recipes[["RecipeId", "Name"]].rename(columns={"RecipeId": "recipe_id"})
names["recipe_id"] = names["recipe_id"].astype("int64")
stats.merge(names, on="recipe_id", how="left").head(10)

global mean rating = 4.637 | m (75th percentile of n_ratings) = 5


,recipe_id,n_interactions,n_ratings,mean_rating,n_liked,weighted_rating,Name
0,186029,58,57,5.000000,57,4.970704,The Best Creole/Cajun Seasoning Mix
1,52285,50,44,5.000000,44,4.962932,Cream Cheese Flan
2,55309,43,43,5.000000,43,4.962160,Caprese Salad Tomatoes (Italian Marinated Tomatoes)
3,205760,41,37,5.000000,37,4.956754,"My ""famous"" Shredded Chicken"
4,141488,36,33,5.000000,33,4.952202,Snickerdoodles
5,91093,33,32,5.000000,32,4.950910,Salt Potatoes New York Style
6,27340,35,31,5.000000,31,4.949546,Pralines
7,26420,99,89,4.966292,89,4.948762,(Shrimp) Cocktail Sauce
8,25703,31,30,5.000000,30,4.948105,Satisfying Cherry Crunch
9,153877,32,29,5.000000,29,4.946578,Perfectly Chocolate Hershey's Hot Cocoa


In [11]:
inter.to_pickle(ALL_PKL)
train.to_pickle(TRAIN_PKL)
test.to_pickle(TEST_PKL)
stats.to_csv(STATS_CSV, index=False, encoding="utf-8-sig")

summary.update({
    "eval_users": int(test["user_id"].nunique()),
    "train_interactions": len(train),
    "test_interactions": len(test),
    "cold_test_items_%": round(cold.mean() * 100, 1),
    "split_mode": SPLIT_MODE,
    "like_threshold": LIKE_THRESHOLD,
    "min_eval": MIN_EVAL,
    "test_frac": TEST_FRAC,
})
pd.Series(summary, name="value").to_csv(SUMMARY_CSV, encoding="utf-8-sig")
print("saved:", ALL_PKL, TRAIN_PKL, TEST_PKL, STATS_CSV, SUMMARY_CSV)

saved: interactions_all.pkl interactions_train.pkl interactions_test.pkl recipe_stats_train.csv interactions_summary.csv
